In [1]:
!pip install -q "dacite>=1.9,<2" --upgrade

!pip install -q transformers datasets evaluate mlflow huggingface_hub accelerate sacrebleu
# Force dagshub to skip checking dependencies
!pip install -q dagshub --no-deps

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 2.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.2/44.2 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 92.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 85.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 43.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 228.4/228.4 kB 16.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 136.5/136.5 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.2/132.2 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.

In [2]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [3]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

# Access Kaggle Secrets
user_secrets = UserSecretsClient()

# Get the Hugging Face token
hf_token = user_secrets.get_secret("HF_TOKEN")

# Authorize Hugging Face
login(token=hf_token)


In [4]:
import mlflow

# Access Kaggle Secrets
user_secrets = UserSecretsClient()

# Pass Dagshub credentials from Kaggle Secrets to Environment Variables
os.environ["MLFLOW_TRACKING_USERNAME"] = user_secrets.get_secret("MLFLOW_USERNAME")
os.environ["MLFLOW_TRACKING_PASSWORD"] = user_secrets.get_secret("MLFLOW_PASSWORD")

# Connect to MLflow server and select the experiment
mlflow.set_tracking_uri("https://dagshub.com/witch2256/pomello_donatello.mlflow")
mlflow.set_experiment("deep-past-initiative-machine-translation")

# Check connection with a test run
with mlflow.start_run(run_name="Participant_3_Setup_Test"):
    mlflow.set_tag("author", "Participant 3")
    mlflow.log_param("architecture", "mBART / mT5 / Subword Seq2Seq")
    mlflow.log_metric("status", 1.0)
    print("Successful MLflow-Dagshub connection!")


Successful MLflow-Dagshub connection!
🏃 View run Participant_3_Setup_Test at: https://dagshub.com/witch2256/pomello_donatello.mlflow/#/experiments/0/runs/7ba2420287b14aa7b42797ad28f2acb3
🧪 View experiment at: https://dagshub.com/witch2256/pomello_donatello.mlflow/#/experiments/0


In [5]:
import pandas as pd
from sklearn.model_selection import train_test_split
from datasets import Dataset, DatasetDict

# 1. Load the data
df = pd.read_csv("/kaggle/input/competitions/deep-past-initiative-machine-translation/train.csv")

# Drop rows with missing values in the clean columns
# df = df.dropna(subset=["transliteration_clean", "translation_clean"]).reset_index(drop=True)
df = df.dropna(subset=["transliteration", "translation"]).reset_index(drop=True)

# 2. Split into train and validation (90/10)
train_df, val_df = train_test_split(df, test_size=0.1, random_state=42)

# Convert to Hugging Face Dataset
dataset = DatasetDict({
    "train": Dataset.from_pandas(train_df),
    "validation": Dataset.from_pandas(val_df)
})

In [6]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, DataCollatorForSeq2Seq, set_seed

# Set global random seed for reproducibility
set_seed(42)

MODEL_NAME = "facebook/mbart-large-50-many-to-many-mmt"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Set language tokens: Akkadian isn't in standard mBART's vocabulary,
# so we reuse an existing language code for both source and target
tokenizer.src_lang = "en_XX"
tokenizer.tgt_lang = "en_XX"

MAX_SOURCE_LENGTH = 256
MAX_TARGET_LENGTH = 256

def preprocess_function(examples):
    ## inputs = examples["transliteration_clean"]
    ## targets = examples["translation_clean"]
    inputs = examples["transliteration"]
    targets = examples["translation"]

    model_inputs = tokenizer(
        inputs,
        max_length=MAX_SOURCE_LENGTH,
        truncation=True,
        padding=False
    )

    labels = tokenizer(
        text_target=targets,
        max_length=MAX_TARGET_LENGTH,
        truncation=True,
        padding=False
    )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_datasets = dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=dataset["train"].column_names
)

model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/529 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/649 [00:00<?, ?B/s]

Map:   0%|          | 0/1404 [00:00<?, ? examples/s]

Map:   0%|          | 0/157 [00:00<?, ? examples/s]

model.safetensors:   0%|          | 0.00/2.44G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/516 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/261 [00:00<?, ?B/s]

In [7]:
import evaluate
import numpy as np

chrf_metric = evaluate.load("chrf")
bleu_metric = evaluate.load("bleu")

def compute_metrics(eval_preds):
    preds, labels = eval_preds
    if isinstance(preds, tuple):
        preds = preds[0]

    # Replace -100 in labels so they decode correctly
    preds = np.where(preds != -100, preds, tokenizer.pad_token_id)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)

    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    decoded_preds = [pred.strip() for pred in decoded_preds]
    decoded_labels = [[label.strip()] for label in decoded_labels]

    chrf_res = chrf_metric.compute(predictions=decoded_preds, references=decoded_labels, word_order=2)  # word_order=2 gives chrF++
    bleu_res = bleu_metric.compute(predictions=decoded_preds, references=decoded_labels)

    return {
        "chrf_pp": round(chrf_res["score"], 4),
        "bleu": round(bleu_res["bleu"] * 100, 4)
    }

In [8]:
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer

HF_REPO_NAME = "AkkadianVoicesCom/Akkadika"
RUN_NAME = "mbart50-clean-finetuning"

set_seed(42)

import mlflow

training_args = Seq2SeqTrainingArguments(
    output_dir="./results_mbart",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=5e-5,
    per_device_train_batch_size=4,        # Increased if VRAM allows
    per_device_eval_batch_size=16,       # Increased from 1 to speed up eval
    gradient_accumulation_steps=2,
    gradient_checkpointing=True,
    optim="adafactor",
    generation_max_length=MAX_TARGET_LENGTH,
    generation_num_beams=1,
    weight_decay=0.01,
    save_total_limit=1,
    num_train_epochs=5,
    predict_with_generate=True,
    fp16=torch.cuda.is_available(),
    logging_steps=10,                    # Log more frequently
    load_best_model_at_end=True,
    metric_for_best_model="chrf_pp",
    greater_is_better=True,
    report_to="none",
    disable_tqdm=False                   # Ensures progress logs fallback to standard output
)

model.gradient_checkpointing_enable()

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)


# Train, logging to MLflow
with mlflow.start_run(run_name=RUN_NAME):
    mlflow.set_tag("author", "Lex")
    mlflow.set_tag("model_architecture", "mbart50")
    mlflow.set_tag("hf_repo", HF_REPO_NAME)

    mlflow.log_params({
        "model_name": MODEL_NAME,
        "lr": training_args.learning_rate,
        "batch_size": training_args.per_device_train_batch_size,
        "epochs": training_args.num_train_epochs,
        "max_src_len": MAX_SOURCE_LENGTH,
        "max_tgt_len": MAX_TARGET_LENGTH,
        "data_columns": "clean"
    })
 
    trainer.train()

    # Evaluate the best model on validation
    eval_metrics = trainer.evaluate()
    mlflow.log_metrics({
        "val_chrF_pp": eval_metrics["eval_chrf_pp"],
        "val_bleu": eval_metrics["eval_bleu"],
        "val_loss": eval_metrics["eval_loss"]
    })

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Chrf Pp,Bleu
1,7.105439,1.721733,37.796000,18.303400
2,5.624686,1.503740,42.734800,21.286100
3,4.335247,1.427299,45.414600,25.997300
4,3.907867,1.417735,45.655600,25.837400
5,3.330831,1.431097,46.512800,26.814900


The following generation flags are not valid and may be ignored: ['early_stopping']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['model.encoder.embed_tokens.weight', 'model.decoder.embed_tokens.weight', 'lm_head.weight'].
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


🏃 View run mbart50-clean-finetuning at: https://dagshub.com/witch2256/pomello_donatello.mlflow/#/experiments/0/runs/d7cdd73519c64ad9b2b0b2c5c39483bd
🧪 View experiment at: https://dagshub.com/witch2256/pomello_donatello.mlflow/#/experiments/0


In [9]:
# Publish the trained model to the Hugging Face Hubx
model.push_to_hub('subword-seq2seq-model', private=True)
tokenizer.push_to_hub("subword-seq2seq-tokenizer", private=True)

README.md: 0.00B [00:00, ?B/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

README.md: 0.00B [00:00, ?B/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

No files have been modified since last commit. Skipping to prevent empty commit.


CommitInfo(commit_url='https://huggingface.co/rillexi/subword-seq2seq-tokenizer/commit/aa0a456a58219cea4933afdd548656e6bd0a94d7', commit_message='Upload tokenizer', commit_description='', oid='aa0a456a58219cea4933afdd548656e6bd0a94d7', pr_url=None, repo_url=RepoUrl('https://huggingface.co/rillexi/subword-seq2seq-tokenizer', endpoint='https://huggingface.co', repo_type='model', repo_id='rillexi/subword-seq2seq-tokenizer'), pr_revision=None, pr_num=None)

In [10]:
# Generate predictions on the validation dataset
raw_predictions = trainer.predict(tokenized_datasets["validation"])
preds = np.where(raw_predictions.predictions != -100, raw_predictions.predictions, tokenizer.pad_token_id)
decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)

val_df_results = pd.DataFrame({
    "id": val_df["oare_id"].values,
    "source_text": val_df["transliteration"].values,
    "target_text": val_df["translation"].values,
    "pred_translation": [p.strip() for p in decoded_preds]
})

val_df_results.to_csv("preds_val_mbart50.csv", index=False)
print("Predictions file preds_val_mbart50.csv saved successfully!")

Predictions file preds_val_mbart50.csv saved successfully!


In [11]:
## Temporary cell to check effective target length
## TO BE DELETED AFTER CHECKING
lengths = [len(tokenizer(t)["input_ids"]) for t in df["translation"]]
import numpy as np
print("p50:", np.percentile(lengths, 50))
print("p95:", np.percentile(lengths, 95))
print("p99:", np.percentile(lengths, 99))
print("max:", max(lengths))

p50: 125.0
p95: 413.0
p99: 701.6000000000004
max: 1321


In [12]:
import os
import pandas as pd
import  torch
from tqdm import tqdm

test_path = "/kaggle/input/competitions/deep-past-initiative-machine-translation/test.csv"

test_df = pd.read_csv(test_path)

model.eval()
model.to("cuda")

# Ensure tokenizer pad token is set
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Get forced BOS token ID to enforce English output in mBART
forced_bos_token_id = tokenizer.lang_code_to_id.get("en_XX")

predictions = []

# Generate outputs row-by-row
for text in tqdm(test_df["transliteration"], desc="Generating Submissions"):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=256).to("cuda")
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs, 
            max_new_tokens=MAX_TARGET_LENGTH,
            pad_token_id=tokenizer.pad_token_id,
            forced_bos_token_id=forced_bos_token_id
        )
    
    # Encoder-decoder models output target tokens directly; decode outputs[0] without slicing
    decoded_output = tokenizer.decode(outputs[0], skip_special_tokens=True).strip()
    predictions.append(decoded_output)

# Form output DataFrame
submission = pd.DataFrame({
    "id": test_df["id"],
    "translation": predictions
})

# Export to root directory without DataFrame index
submission.to_csv("submission.csv", index=False)
print("Saved submission.csv successfully!")

Generating Submissions: 100%|██████████| 4/4 [00:05<00:00,  1.31s/it]

Saved submission.csv successfully!
